# bioeval: open LLMs on public biology benchmarks (WMDP-Bio, LAB-Bench)

Runs on a free Colab T4. **Runtime > Change runtime type > T4 GPU** first.

What this notebook does: (1) installs the project, (2) runs a quick smoke test, (3) runs the full evaluation,
(4) generates the analysis report, (5) zips the results for download. Only answer-letter likelihoods are computed;
no free-form generation, and question text is never saved.

In [ ]:
import torch, subprocess
print('CUDA available:', torch.cuda.is_available())
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)

## 1. Get the code
If you have pushed the repo to GitHub, the clone works. Otherwise upload the project zip with the file browser and run the second option.

In [ ]:
# Option A: clone from GitHub
!git clone https://github.com/blck-iris/bioeval-open-llms.git
%cd bioeval-open-llms

In [ ]:
# Option B: if you uploaded bioeval-open-llms.zip instead, run this and skip Option A
# !unzip -q /content/bioeval-open-llms.zip -d /content
# %cd /content/bioeval-open-llms

In [ ]:
!pip install -q -r requirements.txt

## 2. Tests (no GPU needed, ~5 s)

In [ ]:
!python -m pytest -q

## 3. Smoke test: 50 items per benchmark, one small model (~2 min)
Check that datasets load and the numbers look sane before the long run.

In [ ]:
!python -m bioeval.run_eval --models Qwen/Qwen2.5-0.5B-Instruct \
    --benchmarks wmdp-bio labbench:LitQA2 --limit 50 --out-dir results/smoke --overwrite
!python -m bioeval.analyze --raw-dir results/smoke --out-dir results/smoke

## 4. Full run
Settings used for the write-up: 4 models, WMDP-Bio plus three text-only LAB-Bench subtasks, `plain` and `chat` prompts,
4 cyclic option orders (so the correct answer appears in every position equally often). Expect roughly 1.5 to 3 hours on a T4;
reduce `--n-shifts` to 1 or drop a model to go faster. Results are saved per model, so an interrupted run resumes where it stopped
(existing files are skipped).

In [ ]:
!python -m bioeval.run_eval \
    --models Qwen/Qwen2.5-0.5B-Instruct Qwen/Qwen2.5-1.5B-Instruct HuggingFaceTB/SmolLM2-1.7B-Instruct Qwen/Qwen2.5-3B-Instruct \
    --benchmarks wmdp-bio labbench:LitQA2 labbench:SeqQA labbench:CloningScenarios \
    --templates plain chat --n-shifts 4 --batch-size 8

Optional: a 7B model in 4-bit (needs `bitsandbytes`; Llama and Gemma models are gated, so accept their license on Hugging Face and run `huggingface_hub.login()` first):

In [ ]:
# !python -m bioeval.run_eval --models Qwen/Qwen2.5-7B-Instruct --load-in-4bit --templates plain --n-shifts 4

## 5. Analysis

In [ ]:
!python -m bioeval.analyze --raw-dir results/raw --out-dir results

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/REPORT.md').read()))

In [ ]:
import glob
for f in sorted(glob.glob('results/figures/*.png'))[:6]:
    display(Image(f))

## 6. Download results

In [ ]:
!zip -qr bioeval_results.zip results README.md
from google.colab import files
files.download('bioeval_results.zip')

Next: copy `results/REPORT.md` highlights into the README results section, commit `results/` (small files), and push.
Quote only numbers from your own run.